# Match Outcome Modeling Experiments

This notebook documents the research workflow used to evaluate match outcome models
for the FIFA World Cup 2026 forecasting system.

The objective is to estimate match-level probabilities for:

- win
- draw
- loss

from the perspective of Team A.

These probabilities are later consumed by the Monte Carlo tournament simulator.

This notebook focuses on:

- feature validation
- temporal train/test evaluation
- baseline model assessment
- probabilistic quality diagnostics
- candidate model comparison

## Imports and notebook setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Project paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "match_model_dataset.parquet"

PROJECT_ROOT, DATA_PATH

## Load modeling dataset

The training dataset contains historical international matches with engineered
team-strength and recent-performance features.

In [ ]:
df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()

## Dataset overview

In [ ]:
summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "min_date": df["date"].min(),
    "max_date": df["date"].max(),
    "num_team_a": df["team_a"].nunique() if "team_a" in df.columns else None,
    "num_team_b": df["team_b"].nunique() if "team_b" in df.columns else None,
}

pd.DataFrame([summary])

In [ ]:
df.dtypes.sort_index()

## Target distribution

We inspect class balance before training to understand the baseline difficulty
of the three-class problem.

In [ ]:
target_col = "target"
df[target_col].value_counts(dropna=False)

In [ ]:
target_dist = df[target_col].value_counts(normalize=True).sort_index()
target_dist.plot(kind="bar", figsize=(8, 4))
plt.title("Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Share of matches")
plt.tight_layout()
plt.show()

## Temporal train/test split

To avoid leakage, model evaluation is performed using a chronological split.
Older matches are used for training and newer matches are reserved for testing.

In [ ]:
df = df.sort_values("date").reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"])

split_date = pd.Timestamp("2017-06-14")

train_df = df[df["date"] < split_date].copy()
test_df = df[df["date"] >= split_date].copy()

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print("Train date range:", train_df["date"].min(), "->", train_df["date"].max())
print("Test date range: ", test_df["date"].min(), "->", test_df["date"].max())

## Feature selection

In [ ]:
candidate_features = [
    "team_a_elo_before",
    "team_b_elo_before",
    "elo_diff",
    "abs_elo_diff",
    "team_a_rolling_goals_scored",
    "team_b_rolling_goals_scored",
    "team_a_rolling_goals_conceded",
    "team_b_rolling_goals_conceded",
    "team_a_rolling_goal_diff",
    "team_b_rolling_goal_diff",
    "team_a_rolling_win_rate",
    "team_b_rolling_win_rate",
    "team_a_rolling_points",
    "team_b_rolling_points",
    "neutral_venue",
]

feature_cols = [c for c in candidate_features if c in df.columns]
feature_cols

In [ ]:
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

print(X_train.shape, X_test.shape)

## Missing values and basic feature diagnostics

In [ ]:
missing_summary = (
    X_train.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
)

missing_summary

In [ ]:
X_train.describe().T

## Correlation sanity check

In [ ]:
corr = X_train.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Baseline preprocessing pipeline

In [ ]:
numeric_features = feature_cols

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
    ],
    remainder="drop",
)

## Baseline model: Multiclass Logistic Regression

The baseline model is a multinomial logistic regression trained on engineered
team strength and recent form features.

This model is currently used as the default probability generator for the
simulation engine.

In [ ]:
logreg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        solver="lbfgs",
        max_iter=3000,
        C=1.0,
        random_state=42,
    )),
])

logreg_model.fit(X_train, y_train)

## Baseline predictions and metrics

In [ ]:
y_pred = logreg_model.predict(X_test)
y_proba = logreg_model.predict_proba(X_test)
class_names = list(logreg_model.named_steps["model"].classes_)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Log loss:", round(log_loss(y_test, y_proba, labels=class_names), 4))

In [ ]:
print(classification_report(y_test, y_pred))

## Prediction probability diagnostics

In [ ]:
proba_df = pd.DataFrame(y_proba, columns=class_names)
proba_df.describe()

In [ ]:
proba_df.max(axis=1).hist(bins=30, figsize=(8, 4))
plt.title("Distribution of Maximum Predicted Class Probability")
plt.xlabel("Max predicted probability")
plt.ylabel("Number of matches")
plt.tight_layout()
plt.show()

## Calibration diagnostics

Because simulation quality depends on probability quality rather than just hard
classification accuracy, calibration is an important evaluation criterion.

In [ ]:
def plot_one_vs_rest_calibration(y_true, y_proba_df, positive_class, n_bins=10):
    y_binary = (y_true == positive_class).astype(int)
    prob_true, prob_pred = calibration_curve(
        y_binary,
        y_proba_df[positive_class],
        n_bins=n_bins,
        strategy="uniform",
    )

    plt.figure(figsize=(5, 5))
    plt.plot(prob_pred, prob_true, marker="o", label=positive_class)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title(f"Calibration Curve: {positive_class}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for cls in class_names:
    plot_one_vs_rest_calibration(y_test.reset_index(drop=True), proba_df, cls)

## Candidate model comparison

We compare the logistic regression baseline against a small set of candidate
nonlinear models to assess whether additional complexity improves predictive
quality.

In [ ]:
models = {
    "logistic_regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            solver="lbfgs",
            max_iter=3000,
            C=1.0,
            random_state=42,
        )),
    ]),
    "random_forest": Pipeline(steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[
                ("num", Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]), numeric_features)
            ],
            remainder="drop"
        )),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

In [ ]:
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    classes = list(model.named_steps["model"].classes_)

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, pred),
        "log_loss": log_loss(y_test, proba, labels=classes),
    })

results_df = pd.DataFrame(results).sort_values("log_loss")
results_df

## Feature importance proxy

In [ ]:
logreg_fitted = models["logistic_regression"]
coef_matrix = pd.DataFrame(
    logreg_fitted.named_steps["model"].coef_,
    columns=feature_cols,
    index=logreg_fitted.named_steps["model"].classes_,
)
coef_matrix.T

In [ ]:
coef_matrix.T.plot(kind="bar", figsize=(14, 6))
plt.title("Logistic Regression Coefficients by Class")
plt.ylabel("Coefficient")
plt.tight_layout()
plt.show()

## Error analysis

We inspect a subset of low-confidence and high-confidence predictions to better
understand the model's behavior.

In [ ]:
analysis_df = test_df[["date"]].copy()
analysis_df["y_true"] = y_test.values
analysis_df["y_pred"] = y_pred
analysis_df["max_pred_prob"] = proba_df.max(axis=1).values

for cls in class_names:
    analysis_df[f"proba_{cls}"] = proba_df[cls].values

analysis_df.head()

In [ ]:
analysis_df.sort_values("max_pred_prob", ascending=False).head(10)

In [ ]:
analysis_df.sort_values("max_pred_prob", ascending=True).head(10)

# Conclusions

## Model performance

The multinomial logistic regression baseline achieves an **accuracy of ~0.60** and a **log loss of ~0.87** on the temporal test set.

Considering the inherent unpredictability of international football matches, these results indicate that the model captures a meaningful portion of the signal in the data while maintaining stable probabilistic outputs.

Importantly, the evaluation is performed using a **chronological train/test split**, ensuring that the model is evaluated on matches that occur strictly after the training period. This setup better reflects the real-world forecasting scenario compared to random splits.

---

## Probabilistic quality

Because match outcome probabilities are used as inputs for a **Monte Carlo tournament simulation**, probabilistic quality is more important than raw classification accuracy.

The observed log loss and calibration diagnostics indicate that the logistic regression model produces **well-behaved probability estimates across all three outcome classes**.

The calibration curves show that predicted probabilities closely follow the ideal diagonal line for:

* win
* loss
* draw

This suggests that the model is **well calibrated**, which is critical for downstream simulation tasks.

---

## Class-level performance

The classification report highlights an expected challenge in football outcome modeling: **draw prediction is difficult**.

* Draw recall is very low (~0.01), indicating that the model rarely predicts draws.
* This is common in multiclass football models due to:

  * lower draw frequency
  * weaker predictive signals for stalemates
  * asymmetric outcome distributions.

However, the model performs significantly better in identifying wins and losses:

* win recall: **~0.88**
* loss recall: **~0.62**

Since the simulation engine relies on **probability distributions rather than hard class predictions**, this limitation is less problematic than it would be in a pure classification setting.

---

## Model comparison

A comparison with a Random Forest classifier shows that the logistic regression baseline slightly outperforms the alternative model in terms of **log loss**, which is the most relevant metric for probabilistic forecasting.

| model               | accuracy | log_loss |
| ------------------- | -------- | -------- |
| logistic_regression | ~0.596   | ~0.873   |
| random_forest       | ~0.595   | ~0.883   |

Although tree-based models can capture nonlinear interactions, the logistic regression baseline offers several advantages:

* strong probabilistic calibration
* interpretability
* computational efficiency
* stability when used inside large-scale Monte Carlo simulations.

For these reasons, logistic regression is retained as the **baseline match prediction model used by the tournament simulation engine**.

---

## Implications for the simulation engine

The simulation engine converts match probabilities into tournament outcomes via repeated sampling:

```
predict_match(team_a, team_b)
↓
P(win), P(draw), P(loss)
↓
sample outcome
↓
simulate tournament
```

Because the simulator relies on **probability distributions rather than hard predictions**, a model with stable and calibrated probabilities is preferable to one that simply maximizes classification accuracy.

The logistic regression baseline therefore provides a **robust probabilistic foundation for tournament forecasting**.


---

## Next steps

Planned follow-up work:

1. Tune the logistic regression baseline more systematically.
2. Compare against LightGBM / XGBoost.
3. Evaluate probability calibration more rigorously.
4. Test whether improved match-level metrics translate into improved tournament
   forecast quality.
5. Explore goal-based models as a future extension.